## 👥 07. Subgroup Analysis (데이터 편향 및 하위그룹 검증)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

print('Loading real test predictions...')
try:
    df = pd.read_csv('../checkpoints/test_predictions.csv')
except FileNotFoundError:
    print('⚠️ test_predictions.csv 가 checkpoints 에 없습니다.')
    df = pd.DataFrame()

DISEASES = ["Cardiomegaly", "Effusion", "Hernia"] # 예시

if not df.empty:
    # 1. 성별 분리 연산
    gender_rows = []
    for d in DISEASES:
        prob_col = f'{d}_prob'
        true_col = f'{d}_true'
        if prob_col in df.columns:
            male_df = df[df['Patient Gender'] == 'M']
            fem_df = df[df['Patient Gender'] == 'F']
            
            m_auc = roc_auc_score(male_df[true_col], male_df[prob_col]) if len(male_df[true_col].unique()) > 1 else np.nan
            f_auc = roc_auc_score(fem_df[true_col], fem_df[prob_col]) if len(fem_df[true_col].unique()) > 1 else np.nan
            
            gender_rows.append({
                'Disease': d, 'Male AUROC': m_auc, 'Female AUROC': f_auc,
                'Gap': f"{(m_auc - f_auc) * 100:+.1f}%" if pd.notnull(m_auc) and pd.notnull(f_auc) else "NaN"
            })
            
    if gender_rows:
        g_df = pd.DataFrame(gender_rows)
        g_df.to_csv('../checkpoints/gender_subgroup.csv', index=False)
        print('✅ Saved gender_subgroup.csv')
        display(g_df)
        
    # 2. View Position
    view_rows = []
    for d in ['Cardiomegaly']:
        prob_col = f'{d}_prob'
        true_col = f'{d}_true'
        if prob_col in df.columns:
            pa_df = df[df['View Position'] == 'PA']
            ap_df = df[df['View Position'] == 'AP']
            
            pa_auc = roc_auc_score(pa_df[true_col], pa_df[prob_col]) if len(pa_df[true_col].unique())>1 else np.nan
            ap_auc = roc_auc_score(ap_df[true_col], ap_df[prob_col]) if len(ap_df[true_col].unique())>1 else np.nan
            
            view_rows.append({'View': 'PA', 'N': len(pa_df), 'Mean AUROC': pa_auc, 'Gap vs PA': '—'})
            view_rows.append({'View': 'AP', 'N': len(ap_df), 'Mean AUROC': ap_auc, 
                              'Gap vs PA': f"{(ap_auc-pa_auc)*100:+.1f}%" if pd.notnull(ap_auc) and pd.notnull(pa_auc) else "NaN"})

    if view_rows:
        v_df = pd.DataFrame(view_rows)
        v_df.to_csv('../checkpoints/view_subgroup.csv', index=False)
        print('✅ Saved view_subgroup.csv')
        display(v_df)
